# A simple MODFLOW-USG example

Generate a coarse Voronoi mesh, export a matching DISU/GSF pair, and simulate steady confined flow between two fixed-head sides. The other sides have no flow. There are no wells, faults, recharge or transient stresses.

This uses **MODFLOW-USG**, not MODFLOW 6. Both engines have DISU packages, but their input arrays and connection conventions differ. `to_mfusg_disu()` and `to_mfusgdisu()` target USG only.

Install Python dependencies with `pip install "vorflow[examples]"`. Separately, create a binary directory and run `get-modflow /path/to/bin --subset mfusg`. Add it to PATH or set `MFUSG_EXE` to the executable's full path. The notebook never downloads executables.

In [ ]:
import os
from pathlib import Path
import tempfile

import flopy
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import LineString, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator

## Define and generate the mesh

The rectangle is 200 × 100 m, with a uniform characteristic mesh size of 20 m. Model cell numbers follow the final grid's row positions, not its GeoDataFrame index or generator IDs.

In [ ]:
domain = box(0, 0, 200, 100)
cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1, resolution=20.0)
mg = MeshGenerator(background_lc=20.0, conceptual_mesh=cm)
mg.generate()
vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
gdf = vt.generate()

fig, ax = plt.subplots(figsize=(8, 4))
vt.plot(ax=ax, column=None)
ax.set_aspect("equal")
ax.set_title("Voronoi mesh")
plt.show()

## Build the MODFLOW-USG model

Use one confined layer, top 10 m, bottom 0 m, and hydraulic conductivity 10 m/day. Heads above the layer top are valid for this confined example. FloPy writes the DISU package together with the other model inputs.

The plain-array route is `vt.to_mfusg_disu(top, botm, nlay=1)`. Its `node_id`, `cell_map` and `connectivity` entries are mapping/diagnostic metadata, not FloPy arguments. The package wrapper below removes those entries automatically.

In [ ]:
workspace = tempfile.TemporaryDirectory()
model_ws = Path(workspace.name)
# To retain model.disu, model.gsf and results, replace model_ws with:
# model_ws = Path("mfusg_example").resolve()
# model_ws.mkdir(parents=True, exist_ok=True)

model = flopy.mfusg.MfUsg(
    modelname="model", structured=False,
    model_ws=model_ws, exe_name=os.environ.get("MFUSG_EXE", "mfusg"),
)
top, botm, nlay = 10.0, [0.0], 1
disu = vt.to_mfusgdisu(
    model, top=top, botm=botm, nlay=nlay,
    nper=1, perlen=1.0, nstp=1, steady=True, itmuni=4, lenuni=2,
)
# Use identical geometry/elevations for both files. GSF contains no stresses.
gsf_path = vt.to_gsf(model_ws / "model.gsf", top=top, botm=botm, nlay=nlay)

## Boundary conditions, properties and solver

Cells touching the left side have fixed head 20 m; those touching the right side have fixed head 10 m. Negative `ibound` values mark constant-head cells in MODFLOW-USG. Unassigned outer faces have no flow.

In [ ]:
left = np.flatnonzero(gdf.intersects(LineString([(0, 0), (0, 100)])))
right = np.flatnonzero(gdf.intersects(LineString([(200, 0), (200, 100)])))
assert len(left) and len(right), "Expected cells on both constant-head sides."

ibound = np.ones(len(gdf), dtype=int)
strt = np.full(len(gdf), 15.0)
ibound[left] = ibound[right] = -1
strt[left], strt[right] = 20.0, 10.0
flopy.mfusg.MfUsgBas(model, ibound=ibound, strt=strt)
flopy.mfusg.MfUsgLpf(model, laytyp=0, hk=10.0, vka=10.0, ipakcb=53)
flopy.mfusg.MfUsgSms(
    model, nonlinmeth=0, linmeth=2, clin="cg",  # Symmetric confined flow
    hclose=1e-8, hiclose=1e-8, rclosepcgu=1e-8, mxiter=100, iter1=500,
)
flopy.modflow.ModflowOc(
    model, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]},
    compact=True,
)

## Write, run and check

Require normal termination, finite heads within the two boundary values, and absolute water-budget discrepancy below 0.1%. The simulation does not read GSF; the GSF companion supplies geometry for visualization and external import.

In [ ]:
model.write_input()
assert (model_ws / "model.disu").is_file() and gsf_path.is_file()
success, output = model.run_model(silent=True, report=True)
assert success, "MODFLOW-USG failed:\n" + "\n".join(output)

heads = flopy.utils.HeadUFile(model_ws / "model.hds").get_data()[0].ravel()
assert len(heads) == len(gdf) and np.isfinite(heads).all()
assert heads.min() >= 10.0 - 1e-5 and heads.max() <= 20.0 + 1e-5
budget = flopy.utils.MfusgListBudget(model_ws / "model.list")
incremental, cumulative = budget.get_budget()
assert incremental is not None and len(incremental), "No water budget found."
discrepancy = np.asarray(incremental["PERCENT_DISCREPANCY"])
assert np.isfinite(discrepancy).all() and np.abs(discrepancy).max() < 0.1
print(f"{len(gdf)} cells; maximum budget discrepancy {np.abs(discrepancy).max():.6g}%")

## Plot heads on the original polygons

`cell_map` identifies each model node's zero-based layer and horizontal row. For this single-layer model, assign heads by those row positions, even if the input index or generator IDs are not consecutive.

In [ ]:
mapping = vt.to_mfusg_disu(top, botm, nlay)["cell_map"]
gdf["hydraulic_head"] = heads[mapping.loc[mapping["layer"] == 0, "model_node"].to_numpy()]
fig, ax = plt.subplots(figsize=(9, 4))
gdf.plot(
    column="hydraulic_head", cmap="viridis", edgecolor="0.3", linewidth=0.3,
    legend=True, legend_kwds={"label": "Head (m)"}, ax=ax,
)
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", title="Steady confined flow")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# Remove the temporary files only after inspecting the results.
workspace.cleanup()